# 04 Build Gold Dashboard Tables

Build Power BI-ready Gold tables from Silver data: current snapshot, 5-minute facts, 30-minute and daily aggregates, price events, KPIs, data freshness, and optional interconnector summaries.

## Configure Gold Build Run

This cell creates a run ID and defines price thresholds used for dashboard event flags and price bands.

In [ ]:
# Cell purpose: Configure Gold Build Run.
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window

run_id = str(uuid.uuid4())
high_price_threshold_aud_mwh = 300.0
extreme_price_threshold_aud_mwh = 1000.0

print(f"run_id={run_id}")

## Bootstrap Local Project Package

This cell makes the uploaded `nem_fabric` source package importable in Fabric. Upload `src/nem_fabric` to `Files/libs/nem_fabric` before running the notebook in a Pipeline.

In [ ]:
# Cell purpose: Make nem_fabric importable from Lakehouse Files.
import os
import sys

fabric_lib_path = os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs")
if fabric_lib_path not in sys.path:
    sys.path.insert(0, fabric_lib_path)

print(f"Python library path ready: {fabric_lib_path}")

## Load Silver Source

This cell confirms the required Silver price/demand table exists and loads it as the base for all mandatory Gold tables.

In [ ]:
# Cell purpose: Load Silver Source.
def table_exists(table_name: str) -> bool:
    """Return True when a Lakehouse table exists in the current Spark catalogue."""
    return spark.catalog.tableExists(table_name)


if not table_exists("nem_silver_price_demand_5min"):
    raise RuntimeError("nem_silver_price_demand_5min does not exist. Run notebook 03 first.")

silver = spark.table("nem_silver_price_demand_5min")

## Build Five-Minute Regional Fact

This cell creates the main Power BI fact table with clean column names, price bands, event flags, and one-hour rolling averages.

In [ ]:
# Main 5-minute Gold fact. Flags and bands are precomputed so Power BI stays simple.
rolling_window = Window.partitionBy("region").orderBy(F.col("settlement_datetime").cast("long")).rangeBetween(-3600, 0)

gold_5min = (
    silver
    .withColumn("price_band", F.when(F.col("price_aud_mwh") < 0, "Negative").when(F.col("price_aud_mwh") >= extreme_price_threshold_aud_mwh, "Extreme").when(F.col("price_aud_mwh") >= high_price_threshold_aud_mwh, "High").otherwise("Normal"))
    .withColumn("is_negative_price", F.col("price_aud_mwh") < 0)
    .withColumn("is_high_price", F.col("price_aud_mwh") >= high_price_threshold_aud_mwh)
    .withColumn("is_extreme_price", F.col("price_aud_mwh") >= extreme_price_threshold_aud_mwh)
    .withColumn("rolling_avg_price_1h", F.avg("price_aud_mwh").over(rolling_window))
    .withColumn("rolling_avg_demand_1h", F.avg("demand_mw").over(rolling_window))
    .withColumn("gold_loaded_datetime", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id))
    .select(
        "settlement_datetime", "trading_date", "year", "month", "day", "interval_hour", "interval_minute",
        "region", "region_name", "intervention", "price_aud_mwh", "demand_mw", "available_generation_mw",
        "available_load_mw", "demand_forecast_mw", "dispatchable_generation_mw", "dispatchable_load_mw",
        "net_interchange_mw", "excess_generation_mw", "price_band", "is_negative_price", "is_high_price",
        "is_extreme_price", "rolling_avg_price_1h", "rolling_avg_demand_1h", "gold_loaded_datetime", "run_id"
    )
)

gold_5min.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_gold_region_5min")

## Build Current Snapshot

This cell selects the latest interval for each region to support overview KPI cards and current market status visuals.

In [ ]:
# Current snapshot: latest interval by region for KPI cards.
latest_by_region = Window.partitionBy("region").orderBy(F.col("settlement_datetime").desc())
snapshot = (
    gold_5min.withColumn("row_number", F.row_number().over(latest_by_region))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)
snapshot.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nem_gold_dashboard_current_snapshot")
display(snapshot.orderBy("region"))

## Build Thirty-Minute Regional Aggregate

This cell aggregates 5-minute intervals to 30-minute regional grain for dashboard users who want a less granular view.

In [ ]:
# 30-minute Gold aggregate for interval-granularity switching in Power BI.
gold_30min = (
    gold_5min.withColumn("settlement_30min", F.window("settlement_datetime", "30 minutes").start)
    .groupBy("region", "region_name", "settlement_30min")
    .agg(
        F.avg("price_aud_mwh").alias("price_aud_mwh"),
        F.avg("demand_mw").alias("demand_mw"),
        F.max("price_aud_mwh").alias("max_price_aud_mwh"),
        F.min("price_aud_mwh").alias("min_price_aud_mwh"),
        F.sum(F.col("is_high_price").cast("int")).alias("high_price_interval_count"),
        F.sum(F.col("is_negative_price").cast("int")).alias("negative_price_interval_count"),
    )
    .withColumn("trading_date", F.to_date("settlement_30min"))
    .withColumn("interval_hour", F.hour("settlement_30min"))
    .withColumn("interval_minute", F.minute("settlement_30min"))
)
gold_30min.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_gold_region_30min")

## Build Daily Regional Summary

This cell calculates daily average, maximum, minimum, volatility, demand, and event-count metrics by region.

In [ ]:
# Daily regional summary for trend and volatility pages.
gold_daily = (
    gold_5min.groupBy("region", "region_name", "trading_date")
    .agg(
        F.avg("price_aud_mwh").alias("daily_avg_price"),
        F.max("price_aud_mwh").alias("daily_max_price"),
        F.min("price_aud_mwh").alias("daily_min_price"),
        F.stddev_pop("price_aud_mwh").alias("daily_price_volatility"),
        F.avg("demand_mw").alias("daily_avg_demand"),
        F.max("demand_mw").alias("daily_max_demand"),
        F.sum(F.col("is_high_price").cast("int")).alias("high_price_interval_count"),
        F.sum(F.col("is_extreme_price").cast("int")).alias("extreme_price_interval_count"),
        F.sum(F.col("is_negative_price").cast("int")).alias("negative_price_interval_count"),
    )
)
gold_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_gold_region_daily")

## Build Price Event Table

This cell creates a drill-through table for high, extreme, and negative price intervals.

In [ ]:
# Price events table supports detailed drill-through for spikes and negative intervals.
price_spikes = gold_5min.filter(F.col("is_high_price") | F.col("is_negative_price"))
price_spikes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_gold_price_spikes")

## Build KPIs and Freshness

This cell creates dashboard-level KPI and data freshness tables so Power BI can show operational status without extra transformations.

In [ ]:
# Dashboard KPIs and freshness tables make operational status visible in Power BI.
kpis = (
    snapshot.groupBy()
    .agg(
        F.max("settlement_datetime").alias("latest_settlement_datetime"),
        F.avg("price_aud_mwh").alias("avg_current_price_aud_mwh"),
        F.sum("demand_mw").alias("current_total_demand_mw"),
        F.countDistinct("region").alias("regions_available"),
    )
    .withColumn("run_id", F.lit(run_id))
    .withColumn("gold_loaded_datetime", F.current_timestamp())
)
kpis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nem_gold_dashboard_kpis")

freshness = (
    kpis.select("latest_settlement_datetime", "gold_loaded_datetime", "run_id")
    .withColumn("freshness_minutes", (F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("latest_settlement_datetime")) / 60.0)
    .withColumn("status", F.when(F.col("freshness_minutes") <= 15, "Fresh").when(F.col("freshness_minutes") <= 60, "Delayed").otherwise("Stale"))
    .withColumnRenamed("gold_loaded_datetime", "last_successful_ingestion_datetime")
)
freshness.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nem_gold_data_freshness")
display(kpis)

## Build Optional Interconnector Gold

This cell creates a Power BI-ready interconnector flow table when the Silver interconnector source is available.

In [ ]:
# Optional interconnector Gold table. Built only when notebook 03 created the Silver source.
if table_exists("nem_silver_interconnector_flows"):
    interconnector = spark.table("nem_silver_interconnector_flows")
    interconnector_gold = (
        interconnector
        .withColumn("flow_direction", F.when(F.col("flow_mw") >= 0, "Forward").otherwise("Reverse"))
        .withColumn("interval_hour", F.hour("settlement_datetime"))
        .withColumn("interval_minute", F.minute("settlement_datetime"))
    )
    interconnector_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_gold_interconnector_flows_5min")
else:
    print("Skipping interconnector Gold table because Silver source is unavailable.")